# PFNano Converter Validation

Side-by-side comparison of the input ROOT (PFNano) and output HDF5
to verify PFCands are correctly mapped to `common/tracks` and `cms/pfcands`.

In [1]:
import uproot
import awkward as ak
import numpy as np
import h5py

In [ ]:
# File names
ROOT_FILE = "Tau_PFNano_29-Feb-24_Run2016G-UL2016_MiniAODv2_PFNanoAODv1.root"
H5_FILE   = "pfnano_output.h5"

tree = uproot.open(f"{ROOT_FILE}:Events")
h5 = h5py.File(H5_FILE, "r")

print(f"ROOT events: {tree.num_entries}")
print(f"H5 events:   {h5['metadata'].attrs['n_events']}")
print(f"Input format: {h5['metadata'].attrs['input_format']}")
print(f"has_pfcands:  {h5['metadata'].attrs['has_pfcands']}")

ROOT events: 38963
H5 events:   38963
Input format: PFNanoAOD
has_pfcands:  True


## 1. HDF5 structure overview

In [3]:
def show_shapes(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  {name:50s} {str(obj.shape):>20s}  {obj.dtype}")

for group in ['common', 'cms']:
    print(f"/{group}/")
    h5[group].visititems(show_shapes)
    print()

/common/
  electrons/charge                                            (38963, 20)  float32
  electrons/eta                                               (38963, 20)  float32
  electrons/mask                                              (38963, 20)  bool
  electrons/n                                                    (38963,)  int32
  electrons/phi                                               (38963, 20)  float32
  electrons/pt                                                (38963, 20)  float32
  electrons/trk_iso03                                         (38963, 20)  float32
  event/experiment_id                                            (38963,)  int32
  event/is_simulation                                            (38963,)  int32
  event/mu                                                       (38963,)  float32
  event/pvx                                                      (38963,)  float32
  event/pvy                                                      (38963,)  float32
  ev

## 2. Spot-check: PFCands pT (ROOT vs HDF5)

Compare leading PFCands pT for event 0.

In [4]:
EVT = 0

root_ev = tree.arrays(["PFCands_pt", "PFCands_charge", "PFCands_pdgId",
                       "PFCands_d0", "PFCands_dz"], library="ak",
                      entry_start=EVT, entry_stop=EVT+1)

root_pt = ak.to_numpy(root_ev["PFCands_pt"][0])
root_sorted = np.sort(root_pt[root_pt > 0.2])[::-1]

h5_pt = h5["cms/pfcands/pt"][EVT]
h5_n  = h5["cms/pfcands/n"][EVT]

print(f"Event {EVT}:")
print(f"  ROOT total PFCands: {len(root_pt)}")
print(f"  ROOT with pT > 0.2 GeV: {len(root_sorted)}")
print(f"  H5 stored (max_pfcands=500): {h5_n}")
print()
print(f"  Leading 10 pT (ROOT):  {root_sorted[:10]}")
print(f"  Leading 10 pT (H5):    {h5_pt[:10]}")
print()
n_check = min(h5_n, len(root_sorted))
if np.allclose(root_sorted[:n_check], h5_pt[:n_check], atol=1e-3):
    print("  PASS: pT values match")
else:
    print("  FAIL: pT values do not match")

Event 0:
  ROOT total PFCands: 1160
  ROOT with pT > 0.2 GeV: 995
  H5 stored (max_pfcands=500): 500

  Leading 10 pT (ROOT):  [33.46875   21.8125    20.9375    18.609375  13.6796875 13.1171875
 11.6328125  9.9140625  9.625      8.0078125]
  Leading 10 pT (H5):    [33.46875   21.8125    20.9375    18.609375  13.6796875 13.1171875
 11.6328125  9.9140625  9.625      8.0078125]

  PASS: pT values match


## 3. Spot-check: charged PFCands -> common/tracks

Verify that `common/tracks` contains only charged PFCands with pT > 0.5 GeV.

In [5]:
root_charge = ak.to_numpy(root_ev["PFCands_charge"][0])
charged_mask = (root_charge != 0) & (root_pt > 0.5)
root_charged_pt = np.sort(root_pt[charged_mask])[::-1]

trk_pt = h5["common/tracks/pt"][EVT]
trk_n  = h5["common/tracks/n"][EVT]

print(f"Event {EVT}:")
print(f"  ROOT charged PFCands (pT > 0.5): {len(root_charged_pt)}")
print(f"  H5 common/tracks/n (max=50): {trk_n}")
print()
print(f"  Leading 10 pT (ROOT charged): {root_charged_pt[:10]}")
print(f"  Leading 10 pT (H5 tracks):    {trk_pt[:10]}")
print()
n_check = min(trk_n, len(root_charged_pt))
if np.allclose(root_charged_pt[:n_check], trk_pt[:n_check], atol=1e-3):
    print("  PASS: track pT values match")
else:
    print("  FAIL: track pT values do not match")

Event 0:
  ROOT charged PFCands (pT > 0.5): 213
  H5 common/tracks/n (max=50): 50

  Leading 10 pT (ROOT charged): [18.609375  13.6796875 13.1171875 11.6328125  9.9140625  9.625
  6.1328125  5.5234375  4.7421875  4.6640625]
  Leading 10 pT (H5 tracks):    [18.609375  13.6796875 13.1171875 11.6328125  9.9140625  9.625
  6.1328125  5.5234375  4.7421875  4.6640625]

  PASS: track pT values match


## 4. Spot-check: d0/dz mapping

Verify `PFCands_d0` -> `common/tracks/d0` and `PFCands_dz` -> `common/tracks/z0`.

In [9]:
root_d0 = ak.to_numpy(root_ev["PFCands_d0"][0])
root_dz = ak.to_numpy(root_ev["PFCands_dz"][0])

charged_idx = np.where(charged_mask)[0]
charged_idx_sorted = charged_idx[np.argsort(root_pt[charged_idx])[::-1]]

h5_d0 = h5["common/tracks/d0"][EVT]
h5_z0 = h5["common/tracks/z0"][EVT]

n_show = min(5, trk_n)
print(f"Event {EVT}, first {n_show} tracks:")
print(f"  ROOT d0: {root_d0[charged_idx_sorted[:n_show]]}")
print(f"  H5   d0: {h5_d0[:n_show]}")
print(f"  ROOT dz: {root_dz[charged_idx_sorted[:n_show]]}")
print(f"  H5   z0: {h5_z0[:n_show]}\n")

n_check = min(trk_n, len(charged_idx_sorted))
d0_ok = np.allclose(root_d0[charged_idx_sorted[:n_check]], h5_d0[:n_check], atol=1e-6)
z0_ok = np.allclose(root_dz[charged_idx_sorted[:n_check]], h5_z0[:n_check], atol=1e-6)
print(f"  d0 match: {'PASS' if d0_ok else 'FAIL'}")
print(f"  z0 match: {'PASS' if z0_ok else 'FAIL'}")

Event 0, first 5 tracks:
  ROOT d0: [ 0.00073433 -0.00071526 -0.00139141 -0.00070858 -0.00491333]
  H5   d0: [ 0.00073433 -0.00071526 -0.00139141 -0.00070858 -0.00491333]
  ROOT dz: [-0.00411224 -0.00129986  0.00728607  0.00439072  0.01237488]
  H5   z0: [-0.00411224 -0.00129986  0.00728607  0.00439072  0.01237488]

  d0 match: PASS
  z0 match: PASS


## 5. PFCands pdgId distribution

Check that pdgId values make physical sense.

In [10]:
pdgId = h5["cms/pfcands/pdgId"][:]
mask  = h5["cms/pfcands/mask"][:]
pdg_valid = pdgId[mask]

unique, counts = np.unique(pdg_valid.astype(int), return_counts=True)
print("PFCands pdgId distribution (all events):")
for u, c in sorted(zip(unique, counts), key=lambda x: -x[1]):
    pct = 100.0 * c / len(pdg_valid)
    print(f"  pdgId={u:>4d}: {c:>10d}  ({pct:5.1f}%)")

PFCands pdgId distribution (all events):
  pdgId=   1:    5615389  ( 28.8%)
  pdgId= 211:    4145191  ( 21.3%)
  pdgId=-211:    4013725  ( 20.6%)
  pdgId= 130:    2299591  ( 11.8%)
  pdgId=  22:    1742056  (  8.9%)
  pdgId=   2:    1602951  (  8.2%)
  pdgId= -13:      17869  (  0.1%)
  pdgId=  13:      17415  (  0.1%)
  pdgId=  11:       9349  (  0.0%)
  pdgId= -11:       9125  (  0.0%)


## 6. mask/n consistency check

In [12]:
all_ok = True
for coll in ["common/electrons", "common/muons", "common/jets",
             "common/taus", "common/photons", "common/tracks",
             "cms/pfcands"]:
    n = h5[f"{coll}/n"][:]
    mask = h5[f"{coll}/mask"][:]
    match = np.all(n == mask.sum(axis=1))
    if not match:
        all_ok = False
    print(f"  {coll:30s}  n==sum(mask): {'PASS' if match else 'FAIL'}")

print()
print("All consistent!" if all_ok else "WARNING: mismatches found!")

  common/electrons                n==sum(mask): PASS
  common/muons                    n==sum(mask): PASS
  common/jets                     n==sum(mask): PASS
  common/taus                     n==sum(mask): PASS
  common/photons                  n==sum(mask): PASS
  common/tracks                   n==sum(mask): PASS
  cms/pfcands                     n==sum(mask): PASS

All consistent!


## 7. Physical range checks

In [13]:
all_ok = True
for var_path, mask_path, lo, hi, label in [
    ("cms/pfcands/pt",    "cms/pfcands/mask",    0, 5000, "PFCand pT"),
    ("cms/pfcands/eta",   "cms/pfcands/mask",   -6,    6, "PFCand eta"),
    ("cms/pfcands/phi",   "cms/pfcands/mask", -3.2,  3.2, "PFCand phi"),
    ("common/tracks/pt",  "common/tracks/mask",  0, 5000, "Track pT"),
    ("common/tracks/eta", "common/tracks/mask", -6,    6, "Track eta"),
    ("common/jets/pt",    "common/jets/mask",    0, 5000, "Jet pT"),
    ("common/muons/pt",   "common/muons/mask",   0, 5000, "Muon pT"),
]:
    vals = h5[var_path][:]
    m = h5[mask_path][:]
    v = vals[m]
    if len(v) == 0:
        print(f"  {label:20s}: no entries")
        continue
    ok = np.all((v >= lo) & (v <= hi))
    if not ok:
        all_ok = False
    print(f"  {label:20s}: [{v.min():.3f}, {v.max():.3f}]  expected [{lo}, {hi}]  {'PASS' if ok else 'FAIL'}")

print()
print("All ranges OK!" if all_ok else "WARNING: values out of range!")

  PFCand pT           : [0.200, 1369.000]  expected [0, 5000]  PASS
  PFCand eta          : [-5.120, 5.102]  expected [-6, 6]  PASS
  PFCand phi          : [-3.142, 3.142]  expected [-3.2, 3.2]  PASS
  Track pT            : [0.502, 1369.000]  expected [0, 5000]  PASS
  Track eta           : [-3.414, 2.998]  expected [-6, 6]  PASS
  Jet pT              : [30.016, 2088.000]  expected [0, 5000]  PASS
  Muon pT             : [5.000, 358.531]  expected [0, 5000]  PASS

All ranges OK!


## 8. Verify charged-only in common/tracks

Cross-check 100 events: common/tracks should contain the charged subset of cms/pfcands.

In [14]:
SAMPLE = 100
mismatches = 0

for evt in range(SAMPLE):
    pf_pt     = h5["cms/pfcands/pt"][evt]
    pf_charge = h5["cms/pfcands/charge"][evt]
    pf_mask   = h5["cms/pfcands/mask"][evt]
    trk_pt    = h5["common/tracks/pt"][evt]
    trk_n     = h5["common/tracks/n"][evt]

    valid = pf_mask & (pf_charge != 0) & (pf_pt > 0.5)
    charged_sorted = np.sort(pf_pt[valid])[::-1][:50]

    if trk_n > 0 and len(charged_sorted) > 0:
        n = min(trk_n, len(charged_sorted))
        if not np.allclose(charged_sorted[:n], trk_pt[:n], atol=1e-3):
            mismatches += 1

print(f"Checked {SAMPLE} events: {mismatches} mismatches")
print("PASS: common/tracks contains only charged PFCands" if mismatches == 0 else "FAIL")

Checked 100 events: 0 mismatches
PASS: common/tracks contains only charged PFCands


## Metadata summary

In [15]:
print("Metadata attributes:")
for k, v in sorted(h5['metadata'].attrs.items()):
    print(f"  {k:30s}: {v}")

Metadata attributes:
  converter_changelog           : Add PFNano support: PFCands -> common/tracks (charged) + cms/pfcands (all). Fallback to empty tracks for standard NanoAOD.
  converter_version             : 2.2.0
  electron_eta_cut              : 2.5
  electron_pt_cut_gev           : 7.0
  electron_require_mva_loose    : True
  experiment                    : CMS
  experiment_id                 : 1
  has_pfcands                   : True
  input_file                    : Tau_PFNano_29-Feb-24_Run2016G-UL2016_MiniAODv2_PFNanoAODv1.root
  input_format                  : PFNanoAOD
  jet_eta_cut                   : 4.7
  jet_min_id                    : 2
  jet_pt_cut_gev                : 30.0
  license                       : CC0-1.0
  muon_eta_cut                  : 2.4
  muon_pt_cut_gev               : 5.0
  muon_require_loose            : True
  n_events                      : 38963
  pfcand_pt_cut_all_gev         : 0.2
  pfcand_pt_cut_gev             : 0.5
  photon_pt_cut_gev       

In [16]:
h5.close()